In [5]:
import sys
!{sys.executable} -m pip install duckdb pandas


  Using cached duckdb-1.5.2-cp313-cp313-win_amd64.whl.metadata (4.2 kB)
Using cached duckdb-1.5.2-cp313-cp313-win_amd64.whl (13.1 MB)


In [6]:
import duckdb

In [7]:
import pandas as pd

Data sanity checks

1.Both weeks should show 168 rows (2 platforms × 4 categories × 3 regions × 7 days). Any mismatch = data gap, not a real drop.

In [9]:
query = """
SELECT 
    week, 
    SUM(gmv) AS total_gmv,
    (SUM(orders) * 1.0 / SUM(sessions)) * 100 AS conv_rate
FROM 'gmv_drop_analysis.csv'
GROUP BY week;
"""

In [10]:
df_reults = duckdb.query(query).to_df()
print(df_reults)

           week  total_gmv  conv_rate
0  Mar 30-Apr 5  5444660.0   3.495609
1      Apr 6-12  3977460.0   3.084473


2. Should return zero rows. If not, deduplicate before running any aggregations.

In [17]:
query = """
SELECT
    week,
    COUNT(*)                          AS total_rows,
    COUNT(DISTINCT category)          AS categories,
    COUNT(DISTINCT region)            AS regions,
    COUNT(DISTINCT platform)          AS platforms,
    SUM(CASE WHEN gmv IS NULL
             THEN 1 ELSE 0 END)       AS null_gmv,
    SUM(CASE WHEN orders IS NULL
             THEN 1 ELSE 0 END)       AS null_orders,
    SUM(CASE WHEN sessions IS NULL
             THEN 1 ELSE 0 END)       AS null_sessions
FROM 'gmv_drop_analysis.csv'
GROUP BY week
ORDER BY week;
"""

In [18]:
df_results = duckdb.query(query).to_df()
print(df_results)

           week  total_rows  categories  regions  platforms  null_gmv  \
0      Apr 6-12         168           4        3          2       0.0   
1  Mar 30-Apr 5         168           4        3          2       0.0   

   null_orders  null_sessions  
0          0.0            0.0  
1          0.0            0.0  


3. Flags rows where GMV per order is abnormally high or zero are signs of bad data injection.
GMV Outlier Detection

In [21]:
query = """
SELECT
    date, platform, category, region,
    COUNT(*) AS row_count
FROM 'gmv_drop_analysis.csv'
GROUP BY date, platform, category, region
HAVING COUNT(*) > 1
ORDER BY row_count DESC;
"""

In [22]:
df_results = duckdb.query(query).to_df()
print(df_results)

Empty DataFrame
Columns: [date, platform, category, region, row_count]
Index: []


In [23]:
query = """
SELECT
    date, platform, category, region,
    sessions, orders, gmv,
    ROUND(CAST(gmv AS FLOAT) / NULLIF(orders,0), 0) AS gmv_per_order
FROM 'gmv_drop_analysis.csv'
WHERE
    orders = 0
    OR gmv = 0
    OR ROUND(CAST(gmv AS FLOAT) / NULLIF(orders,0), 0) > 2000
ORDER BY gmv_per_order DESC;
"""

In [24]:
df_results = duckdb.query(query).to_df()
print(df_results)

Empty DataFrame
Columns: [date, platform, category, region, sessions, orders, gmv, gmv_per_order]
Index: []


Confirming the drop

In [25]:
query = """
SELECT
    week,
    SUM(sessions)                                             AS sessions,
    SUM(orders)                                               AS orders,
    SUM(gmv)                                                  AS total_gmv,
    ROUND(SUM(gmv) * 1.0 / SUM(orders), 0)                   AS aov,
    ROUND(SUM(orders) * 100.0 / SUM(sessions), 2)            AS conversion_pct,
    ROUND(SUM(gmv) * 1.0 / SUM(sessions), 2)                 AS revenue_per_session
FROM 'gmv_drop_analysis.csv'
GROUP BY week
ORDER BY week;
"""

In [26]:
df_results = duckdb.query(query).to_df()
print(df_results)

           week  sessions   orders  total_gmv    aov  conversion_pct  \
0      Apr 6-12  664684.0  20502.0  3977460.0  194.0            3.08   
1  Mar 30-Apr 5  670813.0  23449.0  5444660.0  232.0            3.50   

   revenue_per_session  
0                 5.98  
1                 8.12  


1. GMV drops from ~5.44M (Mar 30–Apr 5) to ~3.97M (Apr 6–12). That's a ~27% drop. Confirm conversion rate moved too, or if it's purely an AOV or traffic issue.


In [34]:
query = """
SELECT
    category,
    SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN gmv ELSE 0 END) AS gmv_w1,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN gmv ELSE 0 END) AS gmv_w2,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN gmv ELSE 0 END)
  - SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN gmv ELSE 0 END) AS delta,
    ROUND((SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END)
         - SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END))
        * 100.0
        / SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END), 1) AS wow_pct,
    ROUND((SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END)
         - SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END))
        * 100.0
        / ABS(SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END)
            - SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END)
            + SUM(CASE WHEN week='Apr 6-12' AND category != category THEN 0
                       WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END)), 0
    )                                                          AS dummy,
    ROUND(
      (SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END)
     - SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END))
      * 100.0
      / (SELECT SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END)
               - SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END)
         FROM 'gmv_drop_analysis.csv')
    , 1)                                                       AS share_of_total_drop_pct
FROM 'gmv_drop_analysis.csv'
GROUP BY category
ORDER BY delta ASC;
"""

In [35]:
df_results = duckdb.query(query).to_df()
print(df_results)

        category     gmv_w1     gmv_w2      delta  wow_pct  dummy  \
0    Electronics  4248500.0  2782500.0 -1466000.0    -34.5  -53.0   
1  Home & Garden   402480.0   392880.0    -9600.0     -2.4   -2.0   
2        Fashion   399440.0   401600.0     2160.0      0.5    1.0   
3         Beauty   394240.0   400480.0     6240.0      1.6    2.0   

   share_of_total_drop_pct  
0                     99.9  
1                      0.7  
2                     -0.1  
3                     -0.4  


2. Electronics: delta −1,466,000 (−34.5%). Home & Garden: −9,600 (−2.4%). Fashion and Beauty are flat/positive. Electronics is 99%+ of the total drop.


In [37]:
query = """
SELECT
    week,
    SUM(sessions)                                             AS sessions,
    SUM(orders)                                               AS orders,
    SUM(gmv)                                                  AS gmv,
    ROUND(SUM(gmv) * 1.0 / SUM(orders), 0)                   AS aov,
    ROUND(SUM(orders) * 100.0 / SUM(sessions), 2)            AS conversion_pct,
    ROUND(SUM(gmv) * 1.0 / SUM(sessions), 2)                 AS rev_per_session
FROM 'gmv_drop_analysis.csv'
WHERE category = 'Electronics'
GROUP BY week
ORDER BY week;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

           week  sessions  orders        gmv    aov  conversion_pct  \
0      Apr 6-12  164654.0  5565.0  2782500.0  500.0            3.38   
1  Mar 30-Apr 5  170254.0  8497.0  4248500.0  500.0            4.99   

   rev_per_session  
0            16.90  
1            24.95  


In [38]:
query = """
SELECT
    region,
    SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN gmv ELSE 0 END) AS gmv_w1,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN gmv ELSE 0 END) AS gmv_w2,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN gmv ELSE 0 END)
  - SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN gmv ELSE 0 END) AS delta,
    ROUND((SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END)
         - SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END))
        * 100.0
        / SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END), 1) AS wow_pct
FROM 'gmv_drop_analysis.csv'
WHERE category = 'Electronics'
GROUP BY region
ORDER BY wow_pct ASC;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

          region     gmv_w1    gmv_w2     delta  wow_pct
0           APAC  1429500.0  881500.0 -548000.0    -38.3
1           EMEA  1396500.0  927500.0 -469000.0    -33.6
2  North America  1422500.0  973500.0 -449000.0    -31.6


In [39]:
query = """
SELECT
    date,
    week,
    SUM(gmv)                                                  AS daily_gmv,
    SUM(orders)                                               AS daily_orders,
    ROUND(SUM(gmv) * 1.0 / SUM(orders), 0)                   AS aov,
    ROUND(SUM(orders) * 100.0 / SUM(sessions), 2)            AS conversion_pct
FROM 'gmv_drop_analysis.csv'
WHERE category = 'Electronics'
GROUP BY date, week
ORDER BY date;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

         date          week  daily_gmv  daily_orders    aov  conversion_pct
0  2026-03-30  Mar 30-Apr 5   574500.0        1149.0  500.0            4.99
1  2026-03-31  Mar 30-Apr 5   583500.0        1167.0  500.0            4.99
2  2026-04-01  Mar 30-Apr 5   628500.0        1257.0  500.0            4.99
3  2026-04-02  Mar 30-Apr 5   666000.0        1332.0  500.0            4.99
4  2026-04-03  Mar 30-Apr 5   649500.0        1299.0  500.0            4.99
5  2026-04-04  Mar 30-Apr 5   585500.0        1171.0  500.0            4.99
6  2026-04-05  Mar 30-Apr 5   561000.0        1122.0  500.0            4.99
7  2026-04-06      Apr 6-12   606000.0        1212.0  500.0            4.99
8  2026-04-07      Apr 6-12   601000.0        1202.0  500.0            4.99
9  2026-04-08      Apr 6-12   340000.0         680.0  500.0            2.73
10 2026-04-09      Apr 6-12   267000.0         534.0  500.0            2.47
11 2026-04-10      Apr 6-12   358000.0         716.0  500.0            2.80
12 2026-04-1

Android dropped −49.4% while iOS dropped only −5%. A near-50% drop on one OS while the other stays flat = almost certainly a product/tech issue: a bad app release, checkout bug, or payment failure on Android.


In [40]:
query = """
SELECT
    platform,
    SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN sessions ELSE 0 END) AS sessions_w1,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN sessions ELSE 0 END) AS sessions_w2,
    SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN orders ELSE 0 END)   AS orders_w1,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN orders ELSE 0 END)   AS orders_w2,
    SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN gmv ELSE 0 END)      AS gmv_w1,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN gmv ELSE 0 END)      AS gmv_w2,
    ROUND((SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END)
         - SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END))
        * 100.0
        / SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END), 1) AS gmv_wow_pct,
    ROUND(SUM(CASE WHEN week='Mar 30-Apr 5' THEN orders ELSE 0 END)
         * 100.0
         / SUM(CASE WHEN week='Mar 30-Apr 5' THEN sessions ELSE 0 END), 2) AS conv_w1,
    ROUND(SUM(CASE WHEN week='Apr 6-12' THEN orders ELSE 0 END)
         * 100.0
         / SUM(CASE WHEN week='Apr 6-12' THEN sessions ELSE 0 END), 2)     AS conv_w2
FROM 'gmv_drop_analysis.csv'
GROUP BY platform
ORDER BY gmv_wow_pct ASC;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

  platform  sessions_w1  sessions_w2  orders_w1  orders_w2     gmv_w1  \
0  Android     334696.0     333734.0    11677.0     8997.0  2691860.0   
1      iOS     336117.0     330950.0    11772.0    11505.0  2752800.0   

      gmv_w2  gmv_wow_pct  conv_w1  conv_w2  
0  1361520.0        -49.4     3.49     2.70  
1  2615940.0         -5.0     3.50     3.48  


Android GMV: 2.69M → 1.36M (−49.4%). iOS GMV: 2.75M → 2.61M (−5%). Sessions are similar — so Android users showed up but didn't buy. Classic checkout/payment bug signature.


In [41]:
query = """
SELECT
    platform,
    category,
    SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN gmv ELSE 0 END)   AS gmv_w1,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN gmv ELSE 0 END)   AS gmv_w2,
    ROUND((SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END)
         - SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END))
        * 100.0
        / SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END), 1) AS wow_pct,
    ROUND(SUM(CASE WHEN week='Mar 30-Apr 5' THEN orders ELSE 0 END)
         * 100.0
         / SUM(CASE WHEN week='Mar 30-Apr 5' THEN sessions ELSE 0 END), 2) AS conv_w1,
    ROUND(SUM(CASE WHEN week='Apr 6-12' THEN orders ELSE 0 END)
         * 100.0
         / SUM(CASE WHEN week='Apr 6-12' THEN sessions ELSE 0 END), 2)     AS conv_w2
FROM 'gmv_drop_analysis.csv'
GROUP BY platform, category
ORDER BY platform, wow_pct ASC;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

  platform       category     gmv_w1     gmv_w2  wow_pct  conv_w1  conv_w2
0  Android    Electronics  2092500.0   764000.0    -63.5     4.99     1.83
1  Android  Home & Garden   208320.0   200080.0     -4.0     2.99     2.99
2  Android         Beauty   196000.0   192800.0     -1.6     2.99     2.99
3  Android        Fashion   195040.0   204640.0      4.9     2.99     2.99
4      iOS    Electronics  2156000.0  2018500.0     -6.4     4.99     4.99
5      iOS        Fashion   204400.0   196960.0     -3.6     2.99     2.99
6      iOS  Home & Garden   194160.0   192800.0     -0.7     2.99     2.99
7      iOS         Beauty   198240.0   207680.0      4.8     2.99     2.99


If the Android bug hit all categories equally → it's a platform-level checkout issue. If only Electronics → it may be category-specific (e.g. high-value payment threshold bug).


In [43]:
query = """
SELECT
    date,
    platform,
    SUM(gmv)                                                  AS daily_gmv,
    SUM(orders)                                               AS daily_orders,
    ROUND(SUM(orders) * 100.0 / SUM(sessions), 2)            AS conversion_pct
FROM 'gmv_drop_analysis.csv'
GROUP BY date, platform
ORDER BY date, platform;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

         date platform  daily_gmv  daily_orders  conversion_pct
0  2026-03-30  Android   365500.0        1697.0            3.43
1  2026-03-30      iOS   386120.0        1666.0            3.49
2  2026-03-31  Android   407880.0        1665.0            3.55
3  2026-03-31      iOS   350260.0        1685.0            3.41
4  2026-04-01  Android   371180.0        1663.0            3.46
5  2026-04-01      iOS   430840.0        1763.0            3.55
6  2026-04-02  Android   400880.0        1693.0            3.51
7  2026-04-02      iOS   428240.0        1678.0            3.58
8  2026-04-03  Android   408560.0        1726.0            3.51
9  2026-04-03      iOS   413820.0        1734.0            3.52
10 2026-04-04  Android   359660.0        1582.0            3.48
11 2026-04-04      iOS   397360.0        1733.0            3.49
12 2026-04-05  Android   378200.0        1651.0            3.48
13 2026-04-05      iOS   346160.0        1513.0            3.48
14 2026-04-06  Android   390980.0       

If Android conversion collapses on Apr 6 and stays low all week — it's a persistent bug introduced in a release, not a one-day blip.


In [44]:
query = """
SELECT
    platform,
    week,
    SUM(sessions)                                             AS sessions,
    SUM(orders)                                               AS orders,
    SUM(gmv)                                                  AS gmv,
    ROUND(SUM(orders) * 100.0 / SUM(sessions), 2)            AS conversion_pct,
    ROUND(SUM(gmv) * 1.0 / SUM(orders), 0)                   AS aov,
    ROUND(SUM(gmv) * 1.0 / SUM(sessions), 2)                 AS rev_per_session
FROM 'gmv_drop_analysis.csv'
GROUP BY platform, week
ORDER BY platform, week;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

  platform          week  sessions   orders        gmv  conversion_pct    aov  \
0  Android      Apr 6-12  333734.0   8997.0  1361520.0            2.70  151.0   
1  Android  Mar 30-Apr 5  334696.0  11677.0  2691860.0            3.49  231.0   
2      iOS      Apr 6-12  330950.0  11505.0  2615940.0            3.48  227.0   
3      iOS  Mar 30-Apr 5  336117.0  11772.0  2752800.0            3.50  234.0   

   rev_per_session  
0             4.08  
1             8.04  
2             7.90  
3             8.19  


Sessions barely moved (670K → 664K, −1%). But orders fell 3K and AOV fell ₹38. The funnel broke at the conversion step users arrived but didn't complete purchases, especially on Android.


In [45]:
query = """
SELECT
    platform,
    category,
    ROUND(SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END) * 1.0
         / NULLIF(SUM(CASE WHEN week='Mar 30-Apr 5' THEN sessions ELSE 0 END), 0)
    , 2)                                                       AS rev_per_session_w1,
    ROUND(SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END) * 1.0
         / NULLIF(SUM(CASE WHEN week='Apr 6-12' THEN sessions ELSE 0 END), 0)
    , 2)                                                       AS rev_per_session_w2,
    ROUND((SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END) * 1.0
          / NULLIF(SUM(CASE WHEN week='Apr 6-12' THEN sessions ELSE 0 END), 0))
        - (SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END) * 1.0
          / NULLIF(SUM(CASE WHEN week='Mar 30-Apr 5' THEN sessions ELSE 0 END), 0))
    , 2)                                                       AS delta
FROM 'gmv_drop_analysis.csv'
GROUP BY platform, category
ORDER BY delta ASC;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

  platform       category  rev_per_session_w1  rev_per_session_w2  delta
0  Android    Electronics               24.95                9.13 -15.83
1      iOS    Electronics               24.95               24.93  -0.02
2  Android  Home & Garden                2.39                2.39   0.00
3      iOS  Home & Garden                2.39                2.39   0.00
4  Android        Fashion                2.39                2.39   0.00
5      iOS        Fashion                2.39                2.39   0.00
6  Android         Beauty                2.39                2.39   0.00
7      iOS         Beauty                2.39                2.39   0.00


Android + Electronics show the largest negative delta.


In [46]:
query = """
SELECT
    platform,
    category,
    region,
    SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN gmv ELSE 0 END)      AS gmv_w1,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN gmv ELSE 0 END)      AS gmv_w2,
    SUM(CASE WHEN week = 'Apr 6-12'     THEN gmv ELSE 0 END)
  - SUM(CASE WHEN week = 'Mar 30-Apr 5' THEN gmv ELSE 0 END)      AS delta,
    ROUND((SUM(CASE WHEN week='Apr 6-12' THEN gmv ELSE 0 END)
         - SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END))
        * 100.0
        / SUM(CASE WHEN week='Mar 30-Apr 5' THEN gmv ELSE 0 END), 1) AS wow_pct,
    ROUND(SUM(CASE WHEN week='Mar 30-Apr 5' THEN orders ELSE 0 END)
         * 100.0
         / SUM(CASE WHEN week='Mar 30-Apr 5' THEN sessions ELSE 0 END), 2) AS conv_w1,
    ROUND(SUM(CASE WHEN week='Apr 6-12' THEN orders ELSE 0 END)
         * 100.0
         / SUM(CASE WHEN week='Apr 6-12' THEN sessions ELSE 0 END), 2)     AS conv_w2
FROM 'gmv_drop_analysis.csv'
GROUP BY platform, category, region
ORDER BY delta ASC
LIMIT 5;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

  platform     category         region    gmv_w1    gmv_w2     delta  wow_pct  \
0  Android  Electronics           EMEA  697000.0  225500.0 -471500.0    -67.6   
1  Android  Electronics           APAC  704500.0  265500.0 -439000.0    -62.3   
2  Android  Electronics  North America  691000.0  273000.0 -418000.0    -60.5   
3      iOS  Electronics           APAC  725000.0  616000.0 -109000.0    -15.0   
4      iOS  Electronics  North America  731500.0  700500.0  -31000.0     -4.2   

   conv_w1  conv_w2  
0     4.99     1.70  
1     4.99     1.87  
2     4.99     1.90  
3     4.99     4.99  
4     4.99     4.99  


In [47]:
query = """
WITH platform_week AS (
    SELECT
        platform,
        week,
        SUM(sessions)   AS sessions,
        SUM(orders)     AS orders,
        SUM(gmv)        AS gmv,
        SUM(gmv) * 1.0 / SUM(sessions) AS rev_per_session
    FROM 'gmv_drop_analysis.csv'
    GROUP BY platform, week
),
android_w1 AS (
    SELECT rev_per_session AS baseline_rps
    FROM platform_week
    WHERE platform = 'Android' AND week = 'Mar 30-Apr 5'
),
android_w2 AS (
    SELECT sessions, gmv AS actual_gmv
    FROM platform_week
    WHERE platform = 'Android' AND week = 'Apr 6-12'
)
SELECT
    a2.actual_gmv,
    ROUND(a2.sessions * a1.baseline_rps, 0)             AS expected_gmv_if_no_bug,
    ROUND(a2.sessions * a1.baseline_rps - a2.actual_gmv, 0) AS gmv_lost_to_bug
FROM android_w2 a2, android_w1 a1;
"""

df_results = duckdb.query(query).to_df()
print(df_results)

   actual_gmv  expected_gmv_if_no_bug  gmv_lost_to_bug
0   1361520.0               2684123.0        1322603.0


"Android Electronics in APAC/EMEA/North America all dropped ~50%+ with conversion collapsing. Sessions held steady — users arrived but couldn't complete checkout."
"Root cause: likely a bad Android app release on Apr 6 affecting Electronics checkout flow."
